# Bielik - RAG po polsku

W tym notebooku zbudujemy w pełni lokalny pipeline RAG (Retrieval-Augmented Generation) po polsku - z polskim embedderem [`sdadas/mmlw-retrieval-roberta-large`](https://huggingface.co/sdadas/mmlw-retrieval-roberta-large) i Bielikiem jako generatorem odpowiedzi. Wszystko działa lokalnie, bez wywołań do chmury.

Notebook świadomie nie korzysta z żadnego frameworka RAG-owego (LangChain, LlamaIndex) - używamy minimalnych prymitywów (`sentence-transformers`, `numpy`, klient OpenAI), żeby pokazać, co dzieje się "pod maską". To kontynuacja `010-00. Bielik-podstawy.ipynb`, w którym omówiliśmy chat i structured outputs z Bielikiem.

## Plan notebooka

1. **Załadowanie korpusu**: 50 notatek o miastach z pliku JSON obok notebooka.
2. **Embedder polski `mmlw`**: konfiguracja modelu z poprawnymi prefixami.
3. **Embedowanie korpusu**: cosine similarity w `numpy` (bez Weaviate/FAISS).
4. **Funkcja `retrieve`**: top-k najbardziej pasujących notatek.
5. **Pętla RAG**: retrieval + generacja odpowiedzi przez Bielika z kontekstu.

## Porównanie z sekcją 008

Sekcja 008 (`008-01..03`) pokazuje RAG z OpenAI i Weaviate. Tutaj robimy to samo **w pełni lokalnie i po polsku**, z mniejszą skalą korpusu (50 notatek vs setki), więc zamiast wektorowej bazy danych wystarczy macierz `numpy`.

## Architektura

![Architektura RAG po polsku](assets/010-00.%20Architektura%20RAG.png)

> **Uwaga dot. DevContainera:** Notebook działa wewnątrz kontenera Dockera, więc `localhost` z perspektywy notebooka wskazuje na kontener, a nie hosta z uruchomioną Ollamą. W kodzie używamy `host.docker.internal`, aby z wnętrza kontenera dotrzeć do Ollamy działającej na hoście.

## Wymagania

1. Zainstalowana Ollama: https://ollama.com/download
2. Pobrany model Bielika: `ollama pull SpeakLeash/bielik-11b-v3.0-instruct:bf16`
3. Zbudowany model `bielik-tools` z customowego Modelfile (`010-00. Bielik.Modelfile`):
   ```bash
   ollama create bielik-tools -f "010-00. Bielik.Modelfile"
   ```
   Modelfile naprawia bug ze stop tokens (oryginał używa tokenów Llamy 3 zamiast ChatML, co psuje generację). Szczegóły w `010-00. Bielik.Modelfile.md`.
4. Działający serwis Ollama w tle (port `11434`).
5. Pakiety Pythona: `openai`, `sentence-transformers`, `numpy` (w środowisku Dockera kursu są już zainstalowane).
6. Plik z korpusem `010-00. Bielik-podstawy.json` obok notebooka (jest w repozytorium).

> **Uwaga dot. embeddera:** Pierwsze użycie modelu `sdadas/mmlw-retrieval-roberta-large` pobierze ~1,4 GB wag z HuggingFace. Kolejne uruchomienia korzystają z cache.

## Konfiguracja klienta OpenAI

Używamy tego samego klienta `OpenAI`, którego znamy z `001-01. LLM-OpenAI.ipynb` - tyle, że wskazanego na lokalną Ollamę przez OpenAI-kompatybilny endpoint. To ten sam wzorzec, którego używają wszystkie notebooki sekcji 010.

In [ ]:
import os
import json
from openai import OpenAI

# Klient OpenAI wskazujący na lokalną Ollamę (kompatybilny endpoint /v1).
client = OpenAI(
    base_url="http://host.docker.internal:11434/v1",
    api_key="ollama",  # atrapa - Ollama ignoruje, ale klient OpenAI wymaga niepustego pola
)

# Nazwa modelu zbudowanego z customowego Modelfile.
MODEL = "bielik-tools"

## 1. Załadowanie korpusu

Wczytujemy 50 obiektów `{"city": ..., "note": ...}` z pliku JSON obok notebooka.

In [ ]:
with open("010-00. Bielik-podstawy.json") as f:
    corpus = json.load(f)

print(f"Wczytano {len(corpus)} notatek")
print("\nPrzykładowe rekordy:")
for item in corpus[:3]:
    print(f"  - {item['city']}: {item['note'][:80]}...")

## 2. Embedder polski - `mmlw`

Używamy modelu [`sdadas/mmlw-retrieval-roberta-large`](https://huggingface.co/sdadas/mmlw-retrieval-roberta-large) - polskiego embeddera retrievalowego od Sławomira Dadasa, świetnego w polskich benchmarkach.

Specyfika tego modelu: **wymaga polskiego prefixu `zapytanie: ` dla pytań** użytkownika, natomiast dokumenty w korpusie embedujemy **bez żadnego prefixu**. Asymetria jest świadoma - model był trenowany dokładnie w tym układzie i bez prefixu po stronie zapytań retrieval działa znacząco gorzej.

In [ ]:
from sentence_transformers import SentenceTransformer

# Pierwsze użycie pobierze ~1.4 GB wag z HuggingFace.
embedder = SentenceTransformer("sdadas/mmlw-retrieval-roberta-large")

print(f"Wymiarowość embeddingów: {embedder.get_embedding_dimension()}")

## 3. Embedowanie korpusu

Notatki embedujemy w jednej partii **bez żadnego prefixu** - tak wymaga model `mmlw` po stronie dokumentów. `normalize_embeddings=True` daje wektory długości 1, więc cosine similarity sprowadza się do zwykłego iloczynu skalarnego.

In [ ]:
import numpy as np

# Po stronie dokumentów model mmlw nie wymaga prefixu - embedujemy notatki bezpośrednio.
passage_embs = embedder.encode([item["note"] for item in corpus], normalize_embeddings=True)

print(f"Kształt macierzy embeddingów: {passage_embs.shape}")

## 4. Funkcja `retrieve`

Liczy cosine similarity między pytaniem (z prefixem `zapytanie: `) a wszystkimi notatkami i zwraca `k` najbardziej podobnych. Ponieważ embeddingi są znormalizowane, iloczyn macierz × wektor daje od razu podobieństwa.

In [ ]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    """Zwraca top-k notatek z korpusu najbardziej pasujących do pytania."""
    # Po stronie zapytań mmlw wymaga polskiego prefixu "zapytanie: ".
    query_emb = embedder.encode(f"zapytanie: {query}", normalize_embeddings=True)
    sims = passage_embs @ query_emb
    top_idx = sims.argsort()[-k:][::-1]
    return [corpus[i] for i in top_idx]


# Test retrievalu - same notatki bez wywoływania Bielika.
for item in retrieve("Gdzie znajduje się Wawel?", k=3):
    print(f"  - {item['city']}: {item['note']}")

## 5. Pętla RAG

Łączymy retrieval z generacją: pobieramy top-k notatek, wstrzykujemy je do system promptu jako kontekst i prosimy Bielika o odpowiedź. Model ma wyraźną instrukcję, żeby korzystać **wyłącznie** z dostarczonych notatek - to klasyczne ograniczenie RAG-a, które redukuje halucynacje.

In [ ]:
def rag(question: str, k: int = 3) -> str:
    """RAG: retrieve top-k notatek + odpowiedź Bielika z kontekstem."""
    notes = retrieve(question, k=k)
    context = "\n".join(f"- {n['city']}: {n['note']}" for n in notes)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Odpowiadasz na pytania użytkownika korzystając WYŁĄCZNIE z poniższych notatek "
                    "o miastach. Jeśli odpowiedzi nie ma w notatkach, powiedz że nie wiesz.\n\n"
                    f"NOTATKI:\n{context}"
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content


# Trzy demo zapytania pokazujące działanie pełnego pipeline'u.
pytania = [
    "Gdzie znajduje się Wawel?",
    "Powiedz mi coś ciekawego o Toruniu.",
    "Które miasta z bazy leżą nad Wisłą?",
]

for q in pytania:
    print(f"=== Pytanie: {q} ===")
    print(rag(q))
    print()